# Chapter 5 Lab: Structure Beyond Linearity (`ch04`)

In this lab, we explore different model families (decision trees, k-nearest neighbors, clustering) and learn how to diagnose which family best fits your data's structure. We compare models on synthetic datasets and understand when to switch families.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Seed for reproducibility
np.random.seed(42)

## 1. Decision Trees on Nonlinear Data

We fit decision trees with limited depth (max_depth=3) and unlimited depth to the make_moons dataset. This shows how depth controls overfitting.

In [ ]:
# Generate make_moons data
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)

# Helper function to plot decision boundaries
def plot_decision_boundary(ax, X, y, model, title):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, edgecolors='k', s=30)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

# Fit trees with different depths
tree_shallow = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_shallow.fit(X_moons, y_moons)

tree_deep = DecisionTreeClassifier(max_depth=None, random_state=42)
tree_deep.fit(X_moons, y_moons)

# Plot both
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_decision_boundary(axes[0], X_moons, y_moons, tree_shallow, 'Decision Tree (max_depth=3)')
plot_decision_boundary(axes[1], X_moons, y_moons, tree_deep, 'Decision Tree (Unlimited Depth)')
plt.tight_layout()
plt.show()

print(f"Shallow tree (max_depth=3) accuracy: {tree_shallow.score(X_moons, y_moons):.3f}")
print(f"Deep tree (unlimited) accuracy: {tree_deep.score(X_moons, y_moons):.3f}")
print(f"\nDeep tree has higher training accuracy but likely overfits.")
print(f"Shallow tree generalizes better despite lower training accuracy.")

## 2. k-NN and Distance

We fit k-nearest neighbors with k=1, 5, and 20 on the make_moons dataset. This shows how k controls the bias-variance tradeoff.

In [ ]:
# Generate make_moons data
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)

# Fit k-NN with different k values
knn_1 = KNeighborsClassifier(n_neighbors=1)
knn_1.fit(X_moons, y_moons)

knn_5 = KNeighborsClassifier(n_neighbors=5)
knn_5.fit(X_moons, y_moons)

knn_20 = KNeighborsClassifier(n_neighbors=20)
knn_20.fit(X_moons, y_moons)

# Plot all three
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_decision_boundary(axes[0], X_moons, y_moons, knn_1, 'k-NN (k=1, Overfits)')
plot_decision_boundary(axes[1], X_moons, y_moons, knn_5, 'k-NN (k=5, Balanced)')
plot_decision_boundary(axes[2], X_moons, y_moons, knn_20, 'k-NN (k=20, Underfits)')
plt.tight_layout()
plt.show()

print(f"k=1 accuracy: {knn_1.score(X_moons, y_moons):.3f} (overfitting)")
print(f"k=5 accuracy: {knn_5.score(X_moons, y_moons):.3f} (reasonable)")
print(f"k=20 accuracy: {knn_20.score(X_moons, y_moons):.3f} (underfitting)")
print("\nThe goldilocks value k=5 provides the best balance.")

## 3. When Trees Beat Linear Models

We generate two datasets: one with axis-aligned decision boundaries (trees excel) and one with diagonal boundaries (linear models are fine). This illustrates structural advantage.

In [ ]:
# Dataset 1: Axis-aligned boundaries (two rectangles)
np.random.seed(42)
n_samples = 200
X_axis = np.vstack([
    np.random.uniform(-2, -0.5, (n_samples//2, 2)),
    np.random.uniform(0.5, 2, (n_samples//2, 2))
])
y_axis = np.hstack([np.zeros(n_samples//2), np.ones(n_samples//2)])

# Add noise samples on the boundary
X_axis = np.vstack([X_axis, np.random.uniform(-2, 2, (50, 2))])
y_axis = np.hstack([y_axis, np.random.randint(0, 2, 50)])

# Fit models
lr_axis = LogisticRegression(max_iter=1000, random_state=42)
lr_axis.fit(X_axis, y_axis)

tree_axis = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_axis.fit(X_axis, y_axis)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_decision_boundary(axes[0], X_axis, y_axis, lr_axis, 'Logistic Regression (Struggles)')
plot_decision_boundary(axes[1], X_axis, y_axis, tree_axis, 'Decision Tree (Succeeds)')
plt.tight_layout()
plt.show()

print("Axis-aligned boundaries (two rectangles):")
print(f"  Logistic Regression accuracy: {lr_axis.score(X_axis, y_axis):.3f}")
print(f"  Decision Tree accuracy: {tree_axis.score(X_axis, y_axis):.3f}")
print(f"\n  Trees excel at axis-aligned splits.")

In [ ]:
# Dataset 2: Diagonal boundary
np.random.seed(42)
X_diag = np.random.uniform(-2, 2, (400, 2))
y_diag = (X_diag[:, 0] + X_diag[:, 1] > 0).astype(int)

# Fit models
lr_diag = LogisticRegression(max_iter=1000, random_state=42)
lr_diag.fit(X_diag, y_diag)

tree_diag = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_diag.fit(X_diag, y_diag)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_decision_boundary(axes[0], X_diag, y_diag, lr_diag, 'Logistic Regression (Succeeds)')
plot_decision_boundary(axes[1], X_diag, y_diag, tree_diag, 'Decision Tree (Struggles)')
plt.tight_layout()
plt.show()

print("Diagonal boundary (x1 + x2 > 0):")
print(f"  Logistic Regression accuracy: {lr_diag.score(X_diag, y_diag):.3f}")
print(f"  Decision Tree accuracy: {tree_diag.score(X_diag, y_diag):.3f}")
print(f"\n  Linear models excel at diagonal boundaries.")
print(f"  Trees need many axis-aligned cuts to approximate a diagonal.")

## 4. Clustering as Structure Discovery

We generate three blobs and run k-means with k=2, 3, and 5. Silhouette scores help identify the correct number of clusters.

In [ ]:
# Generate 3 blobs
X_blobs, y_true = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

# Fit k-means with different k
kmeans_2 = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_2.fit(X_blobs)

kmeans_3 = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_3.fit(X_blobs)

kmeans_5 = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_5.fit(X_blobs)

# Compute silhouette scores
sil_2 = silhouette_score(X_blobs, kmeans_2.labels_)
sil_3 = silhouette_score(X_blobs, kmeans_3.labels_)
sil_5 = silhouette_score(X_blobs, kmeans_5.labels_)

# Plot clustering results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, kmeans, k, sil in zip(axes, [kmeans_2, kmeans_3, kmeans_5], [2, 3, 5], [sil_2, sil_3, sil_5]):
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans.labels_, cmap='viridis', s=30, alpha=0.6)
    ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
               c='red', marker='X', s=200, edgecolors='black', linewidths=2)
    ax.set_title(f'k-means (k={k})\nSilhouette={sil:.3f}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print("Silhouette Scores:")
print(f"  k=2: {sil_2:.3f}")
print(f"  k=3: {sil_3:.3f} (best)")
print(f"  k=5: {sil_5:.3f}")
print(f"\nThe highest silhouette score (k=3) correctly identifies the true number of clusters.")

## 5. Structural Diagnosis Workflow

We fit a linear baseline to a difficult dataset (concentric circles), diagnose its failure, and compare alternative model families.

In [ ]:
# Generate concentric circles
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, random_state=42)

# Step 1: Fit linear baseline
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_circles, y_circles)

# Step 2: Check residuals
y_pred_lr = lr.predict(X_circles)
errors = (y_circles != y_pred_lr).astype(int)
error_rate = errors.mean()

# Step 3: Fit alternative models
tree = DecisionTreeClassifier(max_depth=10, random_state=42)
tree.fit(X_circles, y_circles)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_circles, y_circles)

# Plot diagnostic
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Linear baseline
plot_decision_boundary(axes[0, 0], X_circles, y_circles, lr, 'Linear Baseline (Fails)')

# Residual heatmap
h = 0.02
x_min, x_max = X_circles[:, 0].min() - 0.1, X_circles[:, 0].max() + 0.1
y_min, y_max = X_circles[:, 1].min() - 0.1, X_circles[:, 1].max() + 0.1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = lr.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

axes[0, 1].scatter(X_circles[:, 0], X_circles[:, 1], c=errors, cmap='RdYlGn_r', edgecolors='k', s=30)
axes[0, 1].set_title(f'Misclassifications (Error Rate: {error_rate:.1%})')
axes[0, 1].set_xlabel('Feature 1')
axes[0, 1].set_ylabel('Feature 2')

# Decision tree alternative
plot_decision_boundary(axes[1, 0], X_circles, y_circles, tree, 'Decision Tree (Better)')

# k-NN alternative
plot_decision_boundary(axes[1, 1], X_circles, y_circles, knn, 'k-NN (Also Good)')

plt.tight_layout()
plt.show()

print("Structural Diagnosis Workflow:")
print(f"\n1. Linear Baseline:")
print(f"   Accuracy: {lr.score(X_circles, y_circles):.3f}")
print(f"   Error rate: {error_rate:.1%}")
print(f"\n2. Diagnosis:")
print(f"   Errors are concentrated in a ring pattern (radial structure).")
print(f"   The data requires nonlinear decision boundaries.")
print(f"\n3. Alternative Models:")
print(f"   Decision Tree accuracy: {tree.score(X_circles, y_circles):.3f}")
print(f"   k-NN accuracy: {knn.score(X_circles, y_circles):.3f}")
print(f"\n4. Conclusion:")
print(f"   Switch to a nonlinear model family (tree or k-NN).")

## What to Notice

1. **Tree depth controls generalization**: Shallow trees avoid overfitting; deep trees memorize training data.
2. **k-NN has a sweet spot**: k=1 overfits, large k underfits. Use cross-validation to find the right k.
3. **Model families have structural biases**: Trees excel at axis-aligned boundaries, linear models at diagonal ones.
4. **Silhouette scores guide clustering**: They help identify the correct number of clusters without labels.
5. **Residual patterns diagnose failure**: When a baseline model fails, examine where and how it fails. The pattern (e.g., radial, checkerboard) suggests which family to try next.
6. **Structural diagnosis workflow**:
   - Fit a simple baseline (e.g., linear model).
   - Plot predictions and residuals.
   - Identify the misclassification pattern.
   - Try a different model family aligned with that pattern.

Choosing the right model family is as important as tuning hyperparameters. Always start with a simple baseline and use its failures to guide your next choice.